# Playful Activation Steering Demo

This notebook loads an inference-only bundle from Google Drive and runs baseline vs steered generation for the TinyStories ctx512 model.

The heavy training happens on the HPC/NVIDIA side. Colab is the interactive demo layer: change the prompt, layer, alpha, and steering position, then compare the outputs.

## Before Opening In Colab

1. Export the bundle from the repo or cluster:

```bash
sbatch slurm/export_colab_bundle_125m_refined_ctx512.sbatch
```

2. Upload the exported folder to Google Drive, for example:

```text
MyDrive/llm-activation-colab/playful_125m_continue_refined_ctx512/
  model.pt
  tokenizer.json
  vectors.pt
  manifest.json
```

3. Open this notebook in Colab, set `REPO_URL`, mount Drive, and run the cells.

In [ ]:
#@title Setup repo + Drive
REPO_URL = "https://github.com/devinnicholson/llm-activation.git"  #@param {type:"string"}
BUNDLE_DIR = "/content/drive/MyDrive/llm-activation-colab/playful_125m_continue_refined_ctx512"  #@param {type:"string"}

from google.colab import drive
drive.mount("/content/drive")

import pathlib
import subprocess
import sys

repo_dir = pathlib.Path("/content/llm-activation")
if not repo_dir.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(repo_dir)])
else:
    subprocess.check_call(["git", "-C", str(repo_dir), "pull", "--ff-only"])

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_dir)])
src_dir = repo_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
print("repo ready:", repo_dir)
print("src ready:", src_dir)


In [ ]:
#@title Load model, tokenizer, and steering vectors
from pathlib import Path
import json
import sys
import torch

repo_src = Path("/content/llm-activation/src")
if not repo_src.exists():
    raise RuntimeError("Run the setup repo + Drive cell first. Expected /content/llm-activation/src")
if str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

from scratch_llm.activations import register_residual_steering_hook
from scratch_llm.generation import resolve_eos_token_id
from scratch_llm.model import TransformerConfig, TransformerLM
from scratch_llm.tokenizer import ScratchTokenizer

def torch_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

bundle_dir = Path(BUNDLE_DIR)
manifest = json.loads((bundle_dir / "manifest.json").read_text())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

payload = torch_load(bundle_dir / manifest["model"], map_location="cpu")
model = TransformerLM(TransformerConfig(**payload["model_args"]))
model.load_state_dict(payload["model"])
model.to(device)
model.eval()

tokenizer = ScratchTokenizer.from_file(bundle_dir / manifest["tokenizer"], prefer_native=False)
vectors = torch_load(bundle_dir / manifest["vectors"], map_location="cpu")
if "playful" in vectors["vectors"] and "serious" not in vectors["vectors"]:
    vectors["vectors"]["serious"] = {
        layer: -vector for layer, vector in vectors["vectors"]["playful"].items()
    }
stop_token_id = resolve_eos_token_id(tokenizer)

print("device:", device)
print("model args:", payload["model_args"])
print("source iter:", manifest.get("source_iter_num"))
print("source best val loss:", manifest.get("source_best_val_loss"))
print("available vector targets:", sorted(vectors["vectors"].keys()))
print("default demo:", manifest["default_demo"])


In [ ]:
#@title Generation helpers
@torch.no_grad()
def generate_text(prompt, *, max_new_tokens=350, temperature=0.8, top_k=100, seed=1337):
    torch.manual_seed(int(seed))
    ids = tokenizer.encode(prompt, add_special_tokens=False)
    idx = torch.tensor(ids, dtype=torch.long, device=device)[None, :]
    out = model.generate(
        idx,
        max_new_tokens=int(max_new_tokens),
        temperature=float(temperature),
        top_k=int(top_k),
        stop_token_id=stop_token_id,
    )
    return tokenizer.decode(out[0].tolist())


def run_demo(
    prompt,
    *,
    emotion="playful",
    layer=4,
    alpha=1.5,
    position="all",
    max_new_tokens=350,
    temperature=0.8,
    top_k=100,
    seed=1337,
):
    baseline = generate_text(
        prompt,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        seed=seed,
    )

    vector = vectors["vectors"][emotion][int(layer)]
    torch.manual_seed(int(seed))
    handle = register_residual_steering_hook(
        model=model,
        layer=int(layer),
        vector=vector,
        alpha=float(alpha),
        position=position,
    )
    try:
        steered = generate_text(
            prompt,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            seed=seed,
        )
    finally:
        handle.remove()

    return baseline, steered


In [ ]:
#@title Interactive steering demo
import ipywidgets as widgets
from IPython.display import Markdown, clear_output, display

defaults = manifest["default_demo"]
prompt_box = widgets.Textarea(
    value=defaults["prompt"],
    description="Prompt",
    layout=widgets.Layout(width="100%", height="90px"),
)
emotion_dropdown = widgets.Dropdown(
    options=sorted(vectors["vectors"].keys()),
    value=defaults["emotion"],
    description="Vector",
)
layer_slider = widgets.IntSlider(
    value=int(defaults["layer"]),
    min=0,
    max=int(payload["model_args"]["num_layers"]) - 1,
    step=1,
    description="Layer",
)
alpha_slider = widgets.FloatSlider(
    value=float(defaults["alpha"]),
    min=0.0,
    max=12.0,
    step=0.5,
    description="Alpha",
    readout_format=".1f",
)
position_dropdown = widgets.Dropdown(
    options=["all", "last"],
    value=defaults["position"],
    description="Position",
)
tokens_slider = widgets.IntSlider(
    value=int(defaults["max_new_tokens"]),
    min=80,
    max=500,
    step=10,
    description="Tokens",
)
seed_box = widgets.IntText(value=1337, description="Seed")
run_button = widgets.Button(description="Generate", button_style="primary")
output = widgets.Output()

def on_generate(_button):
    with output:
        clear_output(wait=True)
        print("generating...")
        baseline, steered = run_demo(
            prompt_box.value,
            emotion=emotion_dropdown.value,
            layer=layer_slider.value,
            alpha=alpha_slider.value,
            position=position_dropdown.value,
            max_new_tokens=tokens_slider.value,
            seed=seed_box.value,
        )
        clear_output(wait=True)
        display(Markdown(f"### Settings\n`{emotion_dropdown.value}` vector, layer `{layer_slider.value}`, alpha `{alpha_slider.value}`, position `{position_dropdown.value}`"))
        display(Markdown("### Baseline"))
        print(baseline)
        display(Markdown("### Steered"))
        print(steered)

run_button.on_click(on_generate)
display(prompt_box)
display(widgets.HBox([emotion_dropdown, layer_slider, alpha_slider, position_dropdown]))
display(widgets.HBox([tokens_slider, seed_box, run_button]))
display(output)


Suggested default for the ctx512 playful model:

```text
emotion = playful
layer = 4
alpha = 10
position = all
prompt = Once upon a time there was a little robot
```

This is the cleaner setting from the refined continued 125M sweep: strong target movement without the repetitive keyword flooding seen in the highest-alpha layer 4 run. The notebook also exposes `serious` by reversing the playful-vs-serious vector. For a stronger but less coherent playful effect, try `layer = 4`, `alpha = 12`, `position = all`.